In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10,6)
sns.set_palette("Set2")
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import f_oneway
from scipy.stats import kruskal
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    label_binarize
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    ConfusionMatrixDisplay
)

In [11]:
df = pd.read_csv("master_dataset.csv")

In [12]:
df.head()

,Total_Packets,Total_Bytes,Flow_Duration,Mean_Packet_Size,Median_Packet_Size,Std_Packet_Size,Variance_Packet_Size,Min_Packet_Size,Max_Packet_Size,Packet_Size_Range,...,First10_Total_Bytes,First10_Mean_Size,First10_Std_Size,First10_Mean_IAT,First10_Max_IAT,First10_Duration,First10_Forward_Packets,First10_Backward_Packets,First10_Direction_Changes,Label
0,2925,816029,116.347759,278.984273,148.0,331.845078,110121.1560,80,1412,1332,...,6797,679.7,603.798153,0.215236,0.267782,0.267782,3,7,4,Google Doc
1,2813,794628,116.592208,282.484181,148.0,339.031075,114942.0699,80,1412,1332,...,8036,803.6,611.666118,0.521086,0.652493,0.652493,2,8,3,Google Doc
2,2440,714540,116.928839,292.844262,148.0,350.980622,123187.3971,80,1412,1332,...,8032,803.2,612.088523,0.400572,0.500558,0.500558,2,8,3,Google Doc
3,2797,784079,116.729518,280.328566,148.0,335.050275,112258.6868,80,1412,1332,...,8183,818.3,601.057244,0.457744,0.573574,0.573574,2,8,3,Google Doc
4,2504,701820,115.723334,280.279553,148.0,341.916485,116906.8827,80,1412,1332,...,5538,553.8,571.391950,0.162149,0.202761,0.202761,4,6,7,Google Doc


In [13]:
print("Dataset Shape :", df.shape)
print(df.info())

Dataset Shape : (6439, 79)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6439 entries, 0 to 6438
Data columns (total 79 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Total_Packets                  6439 non-null   int64  
 1   Total_Bytes                    6439 non-null   int64  
 2   Flow_Duration                  6439 non-null   float64
 3   Mean_Packet_Size               6439 non-null   float64
 4   Median_Packet_Size             6439 non-null   float64
 5   Std_Packet_Size                6439 non-null   float64
 6   Variance_Packet_Size           6439 non-null   float64
 7   Min_Packet_Size                6439 non-null   int64  
 8   Max_Packet_Size                6439 non-null   int64  
 9   Packet_Size_Range              6439 non-null   int64  
 10  Packet_Size_Q25                6439 non-null   float64
 11  Packet_Size_Q75                6439 non-null   float64
 12  Packet_Size_Q90      

In [14]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Total_Packets,6439.0,6.652927e+03,6.195895e+03,176.000000,902.000000,5.478000e+03,1.073150e+04,4.002100e+04
Total_Bytes,6439.0,7.225949e+06,7.750589e+06,24625.000000,566557.500000,6.774431e+06,1.219748e+07,4.949212e+07
Flow_Duration,6439.0,4.292486e+01,4.035644e+01,0.240002,10.644119,2.472772e+01,6.132976e+01,1.202227e+02
Mean_Packet_Size,6439.0,8.289786e+02,3.799309e+02,135.514019,519.154771,9.164760e+02,1.239807e+03,1.285911e+03
Median_Packet_Size,6439.0,8.101204e+02,6.181530e+02,92.000000,148.000000,1.294000e+03,1.412000e+03,1.486000e+03
...,...,...,...,...,...,...,...,...
First10_Max_IAT,6439.0,9.092889e-01,1.393369e+00,0.000676,0.077541,2.551200e-01,1.866590e+00,1.017260e+01
First10_Duration,6439.0,9.092889e-01,1.393369e+00,0.000676,0.077541,2.551200e-01,1.866588e+00,1.017264e+01
First10_Forward_Packets,6439.0,4.603665e+00,1.444347e+00,1.000000,4.000000,5.000000e+00,6.000000e+00,1.000000e+01
First10_Backward_Packets,6439.0,5.396335e+00,1.444347e+00,0.000000,4.000000,5.000000e+00,6.000000e+00,9.000000e+00


In [15]:
print("Target Classes")
df["Label"].value_counts()

Target Classes


Label
Google Search    1915
Google Drive     1634
Google Doc       1221
Youtube          1077
Google Music      592
Name: count, dtype: int64

In [16]:
X = df.drop(columns=["Label"])
y = df["Label"]
print("Feature Matrix Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Matrix Shape : (6439, 78)
Target Shape : (6439,)


In [17]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Classes :")
for i, cls in enumerate(label_encoder.classes_):
    print(f"{i} : {cls}")

Classes :
0 : Google Doc
1 : Google Drive
2 : Google Music
3 : Google Search
4 : Youtube


In [18]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y_encoded,test_size=0.20,random_state=42,stratify=y_encoded)
print("Training Samples :", X_train.shape)
print("Testing Samples :", X_test.shape)

Training Samples : (5151, 78)
Testing Samples : (1288, 78)


In [20]:
X_train_scaled, X_test_scaled, _, _ = train_test_split(X_scaled,y_encoded,test_size=0.20,random_state=42,stratify=y_encoded)